# Chapter 13: Enhancing Customer Experience — Building a Personalization Engine
**Financial AI in Practice** | Manning Publications

This notebook covers:
- Constructing a "User 360" pipeline to aggregate scattered financial data
- Creating "Financial DNA" vectors using embeddings to capture risk and behavior
- Discovering hidden user "Tribes" via K-Means clustering
- Solving the "Cold Start" problem using semantic matching
- Designing A/B tests and safety guardrails for financial recommendations

**Before you run**
- Data files in this folder (Git LFS files, so run `git lfs pull` after cloning):
  `alphastream_user_mart_20260115.csv` and `etf_metadata_mart_20260115.csv` (listing 13.2), `content_store.pkl` (listing 13.8).
  `user_mart_with_cluster.csv` holds the cluster assignment used in the book's figures (chapter 14 reads it too).
- Listing 13.3 and the bridge cell before listing 13.6 call the OpenAI embeddings API (`text-embedding-3-small`) and need `OPENAI_API_KEY`. Put it in a `.env` file in the repository root (copy `.env.example`; see appendix A.4) or set it as an environment variable. The setup cell below loads the `.env` file.
- Listing 13.1 is a conceptual Airflow DAG. It needs `apache-airflow` with the Apache Spark and Common SQL providers, which the book installs in a separate environment. Nothing later in the notebook depends on it.

Code cells match the printed listings. Cells marked *Not in the book* are small additions needed to run them.

## Setup (not a book listing)
Loads `OPENAI_API_KEY` from a `.env` file in this folder or a parent folder (appendix A.4). Listing 13.3 creates `OpenAI()`, which reads the key from the environment.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
print("OPENAI_API_KEY is set." if os.getenv("OPENAI_API_KEY")
      else "OPENAI_API_KEY is not set: listing 13.3 and the dna_matrix cell will fail.")

## 13.1 The unique context of financial personalization
### 13.1.2 Architecting the pipeline: From raw logs to a User 360 mart
### Listing 13.1 — A conceptual Airflow DAG for the user mart pipeline

This DAG is meant to be deployed to an Airflow environment. Skip this cell if Airflow isn't installed.

In [ ]:
from airflow import DAG
from airflow.providers.apache.spark.operators.spark_sql import SparkSqlOperator
from airflow.providers.common.sql.operators.sql import SQLExecuteQueryOperator
from datetime import datetime

# 1. Define the DAG context
with DAG(
    dag_id='user_behavior_mart_daily_etl',
    start_date=datetime(2026, 1, 1),
    schedule='0 2 * * *',  # Run daily at 2:00 AM after market reconciliation
    catchup=False,
    doc_md="Generates the User Behavior Mart by joining logs and ledger."
) as dag:

    # 2. Task: Aggregate Clickstream Logs (The "Noise")
    # Uses Spark to crunch massive JSON logs from S3 into daily summaries.
    aggregate_clickstream = SparkSqlOperator(
        task_id='aggregate_clickstream_from_s3',
        sql="""
            INSERT OVERWRITE TABLE user_behavior_daily_aggregates
            SELECT
                user_id,
                COUNT(CASE WHEN event = 'login' THEN 1 END) AS login_count_1d,
                SUM(event_duration_sec) AS session_time_1d,
                -- ... (omitted: complex window functions for 15d/30d) ...
            FROM raw_app_logs
            WHERE date = '{{ ds }}'
            GROUP BY user_id;
        """
    )

    # 3. Task: Extract Financial Snapshot (The "Truth")
    # Queries the transactional Postgres database for final EOD balances.
    extract_financial_snapshot = SQLExecuteQueryOperator(
        task_id='extract_financial_snapshot_from_postgres',
        sql="""
            INSERT INTO financial_daily_snapshots
            SELECT
                user_id, total_equity, cash_balance, daily_pnl_pct
            FROM core_ledger_balances
            WHERE snapshot_date = '{{ ds }}';
        """
    )

    # 4. Task: The "User 360" Join
    # Merges the two streams to create the final feature store table.
    build_final_mart = SparkSqlOperator(
        task_id='build_final_user_mart',
        sql="""
            INSERT OVERWRITE TABLE user_behavior_mart
            SELECT
                f.user_id,
                f.total_equity, f.daily_pnl_pct, -- Financial Context
                b.login_count_1d, b.session_time_1d -- Behavioral Context
            FROM financial_daily_snapshots f
            JOIN user_behavior_daily_aggregates b
              ON f.user_id = b.user_id
            WHERE f.snapshot_date = '{{ ds }}';
        """
    )

    # 5. Define Dependencies
    # Logs and Ledger are processed in parallel; the Join waits for both.
    [aggregate_clickstream, extract_financial_snapshot] >> build_final_mart

### 13.1.3 Understanding the AlphaStream dataset
### Listing 13.2 — Loading the user mart and ETF metadata

In [ ]:
import pandas as pd
import ast

# 1. Load the User Behavior Mart (Snapshot date: 2026-01-15)
users = pd.read_csv('alphastream_user_mart_20260115.csv')
users['holdings_list'] = users['holdings_list'].apply(ast.literal_eval)
users['top_search_keywords'] = users['top_search_keywords'].apply(ast.literal_eval)

# 2. Load the ETF Knowledge Base
etfs = pd.read_csv('etf_metadata_mart_20260115.csv')
etfs['tags'] = etfs['tags'].apply(ast.literal_eval)

# 3. Inspect the first few rows
print("User Data loaded:", users.shape)
users.head()

## 13.2 From rules to vectors: Modern user representation
### 13.2.2 Hands-on: Creating the Financial DNA vector with embeddings
### Listing 13.3 — Creating the semantic vectors

Requires `OPENAI_API_KEY`.

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()

# A. Helper to fetch embeddings (In prod, add caching/retries)
def get_embedding(text, model="text-embedding-3-small"):
    return client.embeddings.create(input=[text.replace("\n", " ")], model=model).data[0].embedding

# B. Vectorize the Asset Universe (The "Genes")
etf_vectors = {}
print("Indexing 105 ETFs...")
for index, row in etfs.iterrows():
    # Construct a rich semantic string
    # e.g. "TQQQ: ProShares UltraPro QQQ. Tags: Leveraged Tech High-Risk"
    semantic_string = f"{row['ticker']}: {row['description']} Tags: {' '.join(row['tags'])}"
    etf_vectors[row['ticker']] = np.array(get_embedding(semantic_string))

EMBEDDING_DIM = len(next(iter(etf_vectors.values())))
# C. Logic to synthesize the User Semantic Vector
def create_semantic_vector(user_row):
    vectors = []

    # 1. Portfolio Component (Risk Exposure)
    # In production, weight this loop by Position Value (AUM)
    for ticker in user_row['holdings_list']:
        if ticker in etf_vectors:
            vectors.append(etf_vectors[ticker])

    # 2. Intent Component (Search History)
    # We implicitly weight this by adding it multiple times (w2 > w1)
    if len(user_row['top_search_keywords']) > 0:
        search_text = " ".join(user_row['top_search_keywords'])
        # Note: Embedding on-the-fly incurs latency; cache this in real apps.
        search_vec = np.array(get_embedding(search_text))
        vectors.extend([search_vec] * 2)

    # Edge Case: The "Ghost" User
    # Return small random noise to prevent Divide-by-Zero in cosine similarity
    if not vectors:
        return np.random.normal(0, 0.01, EMBEDDING_DIM)

    return np.mean(vectors, axis=0)

# Apply
sample_users = users.head(10).copy()
sample_users['v_semantic'] = sample_users.apply(create_semantic_vector, axis=1)

### Listing 13.4 — Normalizing behavioral scalars

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Select Features defining "State"
# We include historical trends (6m equity) to capture growth/decline
behavior_cols = [
    'login_count_15d',       # Intensity
    'days_since_signup',     # Maturity
    'total_equity',          # Wealth Scale
    'total_equity_6m_ago',   # Trend Baseline
    'margin_usage_pct'       # Risk Appetite
]

# Handle NaNs (e.g., new users have no 6m history) with 0 or mean
sample_users[behavior_cols] = sample_users[behavior_cols].fillna(0)
# 2. Normalize to [0, 1]
scaler = MinMaxScaler()
# In production, fit this scaler on the entire dataset, not just the batch
scaled_behaviors = scaler.fit_transform(sample_users[behavior_cols])

print(f"Behavioral Vector Size: {scaled_behaviors.shape[1]}")

### Listing 13.5 — Fusing semantics and behavior

In [ ]:
def fuse_vectors(row_index, semantic_vec):
    # Retrieve the scaled behavior vector for this user
    behavior_vec = scaled_behaviors[row_index]

    # Weighting Factor (gamma):
    # Multiplied by 2.0 to ensure Behavior has a strong voice in clustering
    weighted_behavior = behavior_vec * 2.0

    # Concatenate: [1536 floats] + [5 floats] = [1541 floats]
    return np.concatenate([semantic_vec, weighted_behavior])

# Create the final column
sample_users['financial_dna'] = [
    fuse_vectors(i, row['v_semantic'])
    for i, row in sample_users.iterrows()
]

print(f"Final Hybrid DNA Dimension: {len(sample_users.iloc[0]['financial_dna'])}")

### Not in the book: building `dna_matrix` for all 100,000 users
Listings 13.3–13.5 build the Financial DNA for a sample of 10 users. Listing 13.6 then assumes a `dna_matrix` holding the 1,541-dimensional vectors of all 100,000 users ("in a real pipeline, this matrix is loaded from a Vector Database"). The repository doesn't ship those vectors, so this cell computes them with the same functions. Embeddings are cached, so this costs about 130 extra API calls. The matrix takes about 600 MB; with K-means' working copy, expect a peak of about 2.5 GB of RAM in listings 13.6–13.7.

In [ ]:
from functools import lru_cache

# Cache embeddings: there are only ~130 distinct search phrases among 100,000 users.
_get_embedding_uncached = get_embedding

@lru_cache(maxsize=None)
def get_embedding(text, model="text-embedding-3-small"):
    return _get_embedding_uncached(text, model)

np.random.seed(42)  # makes the "ghost user" noise in create_semantic_vector reproducible

# Semantic part (listing 13.3) for every user
dna_matrix = np.zeros((len(users), EMBEDDING_DIM + len(behavior_cols)), dtype=np.float32)
for i, (holdings, keywords) in enumerate(zip(users['holdings_list'], users['top_search_keywords'])):
    dna_matrix[i, :EMBEDDING_DIM] = create_semantic_vector(
        {'holdings_list': holdings, 'top_search_keywords': keywords})

# Behavioral part (listings 13.4-13.5): scaler fitted on the full population, weighted by 2.0
full_scaled_behaviors = MinMaxScaler().fit_transform(users[behavior_cols].fillna(0))
dna_matrix[:, EMBEDDING_DIM:] = full_scaled_behaviors * 2.0

users['financial_dna'] = list(dna_matrix)
print(f"dna_matrix: {dna_matrix.shape}")

### 13.2.3 Discovering tribes: Using clustering to identify hidden user segments
### Listing 13.6 — Uncovering tribes via clustering

K-means cluster numbers depend on the vectors, so your cluster IDs and profiles can differ from figure 13.3. In the book's run, cluster 0 is the Panicked Investors (the "Degen" users of listing 13.8), cluster 1 the Leveraged Holders, cluster 2 the Cold Start Newbies, and cluster 3 the Passive Savers.

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# 1. Scale Up: Run on the full population (100k)
# We assume 'dna_matrix' contains the 1541-dim vectors for all 100,000 users.
# In a real pipeline, this matrix is loaded from a Vector Database.
print(f"Clustering {len(dna_matrix)} users...")

# 2. Run K-Means
# We set k=4 based on our domain hypothesis.
# K-Means is efficient enough to cluster 100k vectors in seconds.
kmeans = KMeans(n_clusters=4, random_state=42)
users['cluster_id'] = kmeans.fit_predict(dna_matrix)

# 3. Interpret the Clusters (The "Reveal")
# We calculate the mean of key features to decode each cluster's "Personality"
analysis_features = [
    'total_equity',
    'margin_usage_pct',
    'pnl_pct_15d',
    'login_count_15d',
    'days_since_signup'
]

# Group by Cluster ID to visualize the profile of each tribe
summary = users.groupby('cluster_id')[analysis_features].mean().round(2)
summary['count'] = users['cluster_id'].value_counts()

print("--- Tribe Personality Profiles ---")
print(summary)

*Not in the book:* the book's cluster assignment is saved in `user_mart_with_cluster.csv`. This cell reproduces the figure 13.3 table from it without any API calls.

In [ ]:
book_users = pd.read_csv('user_mart_with_cluster.csv')
book_features = ['total_equity', 'margin_usage_pct', 'pnl_pct_15d',
                 'login_count_15d', 'days_since_signup']

book_summary = book_users.groupby('cluster_id')[book_features].mean().round(2)
book_summary['count'] = book_users['cluster_id'].value_counts()
print("--- Tribe Personality Profiles (book's cluster assignment) ---")
print(book_summary)

### Listing 13.7 — Visualizing the clusters with t-SNE

Output: figure 13.4.

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Sample 4,000 users for visualization
# We select random INDICES to ensure we pull the matching vectors
sample_indices = np.random.choice(len(users), min(4000, len(users)), replace=False)

viz_sample = users.iloc[sample_indices].copy()  # DataFrame Subset
viz_matrix = dna_matrix[sample_indices]         # Vector Matrix Subset

# 2. Dimensionality Reduction (1541 dims -> 2 dims)
# Perplexity=30 is standard for capturing global structure
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embedding_2d = tsne.fit_transform(viz_matrix)

viz_sample['x'] = embedding_2d[:, 0]
viz_sample['y'] = embedding_2d[:, 1]

# 3. Plot with Accessible Markers
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=viz_sample,
    x='x', y='y',
    hue='cluster_id',
    style='cluster_id',  # Different shapes for accessibility
    palette='viridis',
    s=60,
    alpha=0.7,
    edgecolor='k',
    markers=['o', 's', '^', 'D']
)

plt.title(f'User Tribes Map (Sampled N={len(viz_sample)})')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.legend(title='Cluster ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 13.3 Matching intelligence: From user DNA to content discovery
### 13.3.2 Hands-on: Real-time semantic matching
### Listing 13.8 — The semantic matching engine

Repo note: the vectors in `content_store.pkl` have 1,536 dimensions, while each user's `financial_dna` has 1,541 (listing 13.5 appends five behavioral features). Comparing the full vectors raises a dimension error in `cosine_similarity`, so this cell compares only the user's 1,536-dimensional semantic part. That matches the listing's own comment ("1 sample, 1536 features").

Also note:
- The "Degen" (cluster 0) and "Saver" (cluster 3) users follow the book's cluster numbering. If you ran listing 13.6 yourself, check its profiles to see which cluster is which.
- The four items in `content_store.pkl` carry synthetic stand-in vectors, not `text-embedding-3-small` outputs, so your scores won't match the book's example.

In [ ]:
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# 1. Setup: Load the "Mock Vector DB"
# (In a real app, this would be a connection to Pinecone/Milvus)
# We assume 'content_store.pkl' contains the list of vectorized articles generated in the pipeline.
with open("content_store.pkl", "rb") as f:
    content_store = pickle.load(f)

def get_recommendations(user_id, content_list, top_k=2):
    # 2. Fetch the User Vector (Pre-computed Financial DNA)
    user_row = users.loc[users['user_id'] == user_id]
    if user_row.empty:
        return []

    # Reshape for scikit-learn (1 sample, 1536 features)
    user_vec = np.array(user_row.iloc[0]['financial_dna'][:1536]).reshape(1, -1)  # repo: semantic part only, see note above

    # 3. Score Content Alignment
    results = []
    for item in content_list:
        content_vec = np.array(item['vector']).reshape(1, -1)
        # Cosine Similarity: 1.0 = Perfect Match, 0.0 = No Relation
        score = cosine_similarity(user_vec, content_vec)[0][0]
        results.append((item['title'], item['type'], score))

    # 4. Rank by Relevance
    return sorted(results, key=lambda x: x[2], reverse=True)[:top_k]

# --- Test Scenario: Comparing Tribes ---
# We select representative users from our clusters to validate the logic.
degen_user = users.loc[users['cluster_id'] == 0, 'user_id'].iloc[0] # Cluster 0: Degen
saver_user = users.loc[users['cluster_id'] == 3, 'user_id'].iloc[0] # Cluster 3: Saver

print(f"--- Recommendations for Degen User ({degen_user}) ---")
for title, ctype, score in get_recommendations(degen_user, content_store):
    print(f"[{ctype}] {title} (Score: {score:.3f})")

print(f"\n--- Recommendations for Saver User ({saver_user}) ---")
for title, ctype, score in get_recommendations(saver_user, content_store):
    print(f"[{ctype}] {title} (Score: {score:.3f})")

### Not in the book: score against real content embeddings (optional)

The four items in `content_store.pkl` carry stand-in vectors. This cell re-embeds each item's title and tags with the same `get_embedding` helper from listing 13.3 (four API calls) and reruns the two recommendations, so the scores reflect actual semantic similarity.

In [ ]:
content_store_real = [
    {**item, "vector": np.array(get_embedding(f"{item['title']} Tags: {' '.join(item['tags'])}"))}
    for item in content_store
]

for label, uid in [("Degen", degen_user), ("Saver", saver_user)]:
    print(f"--- {label} user ({uid}), real content embeddings ---")
    for title, ctype, score in get_recommendations(uid, content_store_real):
        print(f"[{ctype}] {title} (Score: {score:.3f})")
